# MAPPO Exploration Curriculum

Train the same feed-forward, local-vision JAX MAPPO ant to explore the full grid. The actor keeps the compatible AntByte observation shape, but the training reward is the number of newly visited distinct cells; cookies and byte writes do not drive this setup.


In [ ]:
from pathlib import Path
import os
import sys

# Set these before importing JAX in this kernel.
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.35")
if "jax" in sys.modules:
    print("Restart the kernel before rerunning training; JAX was already imported.")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from ant_byte_env import notebook_workflows as workflows

runtime_status = workflows.configure_jax_notebook_runtime()
workflows.assert_notebook_resources_available(runtime_status)
runtime_status


In [ ]:
import importlib

import jax

from ant_byte_env.training.jax_mappo import runner as jax_runner

workflows = importlib.reload(workflows)
jax_runner = importlib.reload(jax_runner)
print(f"JAX device: {jax.devices()[0]}")


## Quick Smoke Run

Run one tiny training job to confirm the kernel, package imports, and JAX path are wired correctly.


In [ ]:
smoke_metrics = workflows.run_jax_smoke(jax_runner.main)
smoke_metrics


## Curriculum Settings

Edit only the stage sizes or high-level run constants here. The exploration objective and CLI argument plumbing live in `ant_byte_env.notebook_workflows`.


In [ ]:
RUN_DIR = PROJECT_ROOT / "runs" / "notebooks" / "exploration_curriculum"
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
MEDIA_DIR = RUN_DIR / "media"

STAGE_SIZES = range(4, 51)
NUM_ENVS = 16
ROLLOUT_TILE_SIZE = workflows.NOTEBOOK_ROLLOUT_TILE_SIZE
ACTOR_VISION_RADIUS = 1
WRITE_BITS = 1
WANDB_PROJECT = None
WANDB_ENTITY = None
WANDB_GROUP = "exploration_curriculum_50x50"
WANDB_MODE = "online"
WANDB_VIDEO_MAX_FRAMES = 600
WANDB_VIDEO_STAGE_NAMES = workflows.EXPLORATION_WANDB_PREVIEW_STAGE_NAMES

CURRICULUM_STAGES = workflows.build_exploration_curriculum_stages(STAGE_SIZES)
GLOBAL_UPDATE_CAP = max(int(stage["global_update_cap"]) for stage in CURRICULUM_STAGES)
FIRST_STAGE_NUM_STEPS = int(CURRICULUM_STAGES[0]["num_steps"])
UPDATE_TIMESTEPS = workflows.update_timesteps(
    num_envs=NUM_ENVS,
    num_steps=FIRST_STAGE_NUM_STEPS,
)
COMMON_ARGS = workflows.build_exploration_common_args(
    CURRICULUM_STAGES,
    num_envs=NUM_ENVS,
    num_steps=FIRST_STAGE_NUM_STEPS,
    actor_vision_radius=ACTOR_VISION_RADIUS,
    write_bits=WRITE_BITS,
)
CURRICULUM_STAGES[:2], CURRICULUM_STAGES[-1], COMMON_ARGS


## Train Curriculum Checkpoints


In [ ]:
exploration_result = workflows.run_exploration_curriculum(
    stages=CURRICULUM_STAGES,
    checkpoint_dir=CHECKPOINT_DIR,
    common_args=COMMON_ARGS,
    update_timesteps_per_stage=UPDATE_TIMESTEPS,
    global_update_cap=GLOBAL_UPDATE_CAP,
    train_main=jax_runner.main,
    wandb_project=WANDB_PROJECT,
    wandb_entity=WANDB_ENTITY,
    wandb_group=WANDB_GROUP,
    wandb_mode=WANDB_MODE,
    wandb_video_max_frames=WANDB_VIDEO_MAX_FRAMES,
    wandb_video_stage_names=WANDB_VIDEO_STAGE_NAMES,
)
FINAL_CHECKPOINT = exploration_result["final_checkpoint_path"]
exploration_result


## Optional Render and Vault


In [ ]:
rollout_result = workflows.render_exploration_rollouts(
    run_dir=RUN_DIR,
    checkpoint_dir=CHECKPOINT_DIR,
    media_dir=MEDIA_DIR,
    stages=CURRICULUM_STAGES,
    stage_names=WANDB_VIDEO_STAGE_NAMES,
    actor_vision_radius=ACTOR_VISION_RADIUS,
    write_bits=WRITE_BITS,
    global_update_cap=GLOBAL_UPDATE_CAP,
    tile_size=ROLLOUT_TILE_SIZE,
)
rollout_result
